# Generate RecBole Atomic Files

Converts the Amazon Reviews 2023 gzipped JSONL data (reviews + meta) into RecBole Atomic Files (.inter).

**Output structure:**  
`dataset/{dataset_name}/{dataset_name}.inter`  

In [7]:
import json
import gzip
from pathlib import Path
from typing import Any
import pandas as pd

In [8]:
# --- Config ---
DATASET_NAME: str = "Beauty_and_Personal_Care"
CATEGORIES: list[str] = ["Beauty_and_Personal_Care"]
SAMPLE_SIZE: int | None = 1_000_000            # None = all rows
DATA_DIR: str = "../data"

# Only include reviews between these dates (inclusive). Set to None to skip filtering.
START_DATE: str | None = "2020-01-01"
END_DATE: str | None = "2022-12-31"


## Common utils

In [9]:
def stream_jsonl(path: str, fields: list[str] | None = None):
    with gzip.open(path, "rt", encoding="utf-8") as f:
        for _, line in enumerate(f):
            obj = json.loads(line)
            if fields is not None:
                obj = {k: obj.get(k) for k in fields}
            yield obj

def _date_to_ms(date_str: str | None) -> int | None:
    if date_str is None:
        return None
    return int(pd.Timestamp(date_str, tz="UTC").timestamp() * 1000)

def load_reviews(
    categories: list[str], sample_size: int | None = None, 
    start_date: str | None = None, end_date: str | None = None, 
    min_rating: int | None = None
) -> list[dict[str, Any]]:
    start_ts = _date_to_ms(start_date)
    end_ts = _date_to_ms(end_date)
    print(f"Filtering reviews with criteria: start_date={start_date}, end_date={end_date}, min_rating={min_rating}")

    reviews: list[dict[str, Any]] = []
    for cat in categories:
        path = f"{DATA_DIR}/{cat}.jsonl.gz"
        print(f"Loading reviews: {path}")
        for obj in stream_jsonl(path, fields=[
            'user_id', 'parent_asin', 'rating', 'timestamp'
        ]):
            ts: Any = obj.get("timestamp")
            rating: Any = obj.get("rating")
            if start_ts is not None and (ts is not None and ts < start_ts):
                continue
            if end_ts is not None and (ts is not None and ts > end_ts):
                continue
            if min_rating is not None and (rating is not None and rating < min_rating):
                continue
            obj["category"] = cat
            reviews.append(obj)

            if sample_size is not None and len(reviews) >= sample_size:
                print(f"Reached sample size limit ({sample_size} reviews). Stopping.")
                break
    return reviews


## Load reviews

In [10]:
reviews: list[dict[str, Any]] = load_reviews(CATEGORIES, sample_size=SAMPLE_SIZE, start_date=START_DATE, end_date=END_DATE)
print(f"Loaded {len(reviews):,} reviews")

Filtering reviews with criteria: start_date=2020-01-01, end_date=2022-12-31, min_rating=None
Loading reviews: ../data/Beauty_and_Personal_Care.jsonl.gz
Reached sample size limit (1000000 reviews). Stopping.
Loaded 1,000,000 reviews


In [11]:
# Build unique user and item ID sets
user_ids: set[str] = set()
item_ids: set[str] = set()
for r in reviews:
    user_ids.add(r["user_id"])
    item_ids.add(r["parent_asin"])

user_map: dict[str, int] = {uid: i for i, uid in enumerate(sorted(user_ids))}
item_map: dict[str, int] = {pid: i for i, pid in enumerate(sorted(item_ids))}
print(f"Users: {len(user_map):,}  Items: {len(item_map):,}")

Users: 330,737  Items: 209,389


### Write .inter file (user-item interactions)

In [12]:
out_dir: Path = Path(DATA_DIR) / DATASET_NAME
out_dir.mkdir(parents=True, exist_ok=True)
inter_path: Path = out_dir / f"{DATASET_NAME}.inter"

with open(inter_path, "w") as f:
    f.write("user_id:token\titem_id:token\trating:float\ttimestamp:float\n")
    for r in reviews:
        uid: int | None = user_map.get(r["user_id"])
        iid: int | None = item_map.get(r["parent_asin"])
        if uid is None or iid is None:
            continue
        f.write(f"{uid}\t{iid}\t{r['rating']}\t{r['timestamp']}\n")

print(f"Wrote {inter_path}")

Wrote ../data/Beauty_and_Personal_Care/Beauty_and_Personal_Care.inter
